# NeuroBridge-S4 Graph Learning — Phase 10: HRP-Like Data Adapter Layer

> Phase 10 does not interpret health status. It validates and transforms HRP-like longitudinal data streams into a graph-ready self-baseline schema for downstream NeuroBridge-S4 analysis.

```text
raw / semi-structured HRP-like streams
  -> schema validation
  -> variable-to-domain mapping
  -> self-baseline transformation
  -> domain-level longitudinal scores
  -> graph-ready NeuroBridge-S4 inputs
```

## 2. Plain-language purpose

Phase 10 adds a reusable adapter layer that transforms HRP-like longitudinal data streams into the graph-ready domain score format used by the NeuroBridge-S4 pipeline. It separates *data ingestion and validation* from *graph analysis*.

## 3. Why this matters

Real biomedical / human research data rarely arrive as clean graph-ready domain tables. They often arrive as separate biomarker, sleep/activity, cognitive, questionnaire, and environmental tables. Phase 10 bridges raw data streams and the longitudinal graph trajectory pipeline (Phase 6).

## 4. Scientific guardrails

- This is **not** actual Artemis II data unless explicitly provided.
- Template data are **schema examples only** (`schema_template_not_scientific_evidence`).
- The adapter does **not** diagnose.
- The adapter does **not** score health risk.
- The adapter does **not** infer exposure.
- The adapter does **not** recommend treatment.
- Variable-to-domain mapping is an interpretation scaffold for downstream analysis, not a clinical judgment.

In [1]:
import sys
from pathlib import Path

repo_root = Path().resolve().parent
if str(repo_root / 'src') not in sys.path:
    sys.path.insert(0, str(repo_root / 'src'))

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 160)

from neurobridge_graph.data_adapters import (
    create_data_templates,
    standardize_wide_longitudinal_table,
    standardize_long_longitudinal_table,
    combine_standardized_streams,
    compute_variable_baseline_deltas,
    build_domain_scores_from_variables,
    pivot_domain_scores_wide,
    run_adapter_pipeline,
)
from neurobridge_graph.data_validation import (
    detect_table_format, build_input_readiness_report,
)
from neurobridge_graph.domain_mapping import (
    get_default_variable_domain_mapping, map_variables_dataframe,
    build_domain_coverage_report,
)

RESULTS_TABLES = repo_root / 'results' / 'tables'
RESULTS_REPORTS = repo_root / 'results' / 'reports'
TEMPLATES_DIR = repo_root / 'data' / 'templates'
HRP_INPUTS_DIR = repo_root / 'data' / 'hrp_like_inputs'
print('Environment ready.')

Environment ready.


## 5. Create and inspect templates

Templates are written to `data/templates/`. Each row is marked `schema_template_not_scientific_evidence`.

In [2]:
template_paths = create_data_templates(TEMPLATES_DIR)
for p in template_paths:
    df = pd.read_csv(p)
    print(f'{p.name}: {list(df.columns)}')

longitudinal_wide_template.csv: ['subject_id', 'timepoint', 'mission_phase', 'time_index', 'heart_rate', 'hrv_rmssd', 'sleep_duration', 'reaction_time', 'crp', 'glucose', 'data_type']
longitudinal_long_template.csv: ['subject_id', 'timepoint', 'mission_phase', 'time_index', 'variable_name', 'value', 'unit', 'data_stream', 'data_type']
biomarkers_template.csv: ['subject_id', 'timepoint', 'mission_phase', 'time_index', 'crp', 'glucose', 'hemoglobin', 'hematocrit', 'cholesterol', 'data_type']
sleep_activity_template.csv: ['subject_id', 'timepoint', 'mission_phase', 'time_index', 'sleep_duration', 'sleep_efficiency', 'wake_after_sleep_onset', 'steps', 'resting_hr', 'data_type']
cognitive_tests_template.csv: ['subject_id', 'timepoint', 'mission_phase', 'time_index', 'reaction_time', 'accuracy', 'cognitive_score', 'data_type']
questionnaires_template.csv: ['subject_id', 'timepoint', 'mission_phase', 'time_index', 'fatigue_score', 'stress_score', 'mood_score', 'recovery_score', 'data_type']
e

## 6. Load HRP-like input files

The notebook looks for real input files in `data/hrp_like_inputs/`. If none exist, it uses the generated templates as schema demonstration only.

In [3]:
input_paths = []
if HRP_INPUTS_DIR.exists():
    input_paths = sorted(HRP_INPUTS_DIR.glob('*.csv'))

if input_paths:
    print(f'Found {len(input_paths)} real HRP-like input file(s).')
    for p in input_paths:
        print(' -', p.name)
else:
    print('No real HRP-like input files were found. The notebook will use '
          'schema templates to demonstrate adapter mechanics only.')

No real HRP-like input files were found. The notebook will use schema templates to demonstrate adapter mechanics only.


## 7. Validate, standardize, map, and build domain scores

`run_adapter_pipeline` performs the full flow: load -> validate -> detect format -> standardize -> map variables -> self-baseline deltas -> domain scores -> export reports and graph-ready outputs. Below we also display the intermediate artifacts.

In [4]:
outputs = run_adapter_pipeline(
    input_paths, output_dir=RESULTS_TABLES, templates_dir=TEMPLATES_DIR)
for name, path in outputs.items():
    print(f'{name} -> {path.relative_to(repo_root)}')

adapter_input_readiness_report.csv -> results/tables/adapter_input_readiness_report.csv
standardized_longitudinal_variables.csv -> results/tables/standardized_longitudinal_variables.csv
variable_baseline_deltas.csv -> results/tables/variable_baseline_deltas.csv
variable_domain_mapping_report.csv -> results/tables/variable_domain_mapping_report.csv
domain_coverage_report.csv -> results/tables/domain_coverage_report.csv
adapter_domain_scores_long.csv -> results/tables/adapter_domain_scores_long.csv
adapter_domain_scores_wide.csv -> results/tables/adapter_domain_scores_wide.csv
adapter_generated_longitudinal_domain_scores.csv -> results/tables/adapter_generated_longitudinal_domain_scores.csv
phase10_data_adapter_report.txt -> results/reports/phase10_data_adapter_report.txt


### 7a. Input readiness report

Saved to `results/tables/adapter_input_readiness_report.csv`.

In [5]:
pd.read_csv(RESULTS_TABLES / 'adapter_input_readiness_report.csv')

,table_name,detected_format,rows,columns,subject_count,timepoint_count,required_columns_status,missingness_summary,notes
0,biomarkers_template.csv,wide_longitudinal,2,10,1,2,ok,0 columns with missing values (mean missing fr...,ok
1,cognitive_tests_template.csv,wide_longitudinal,2,8,1,2,ok,0 columns with missing values (mean missing fr...,ok
2,environmental_context_template.csv,wide_longitudinal,2,10,1,2,ok,0 columns with missing values (mean missing fr...,ok
3,longitudinal_long_template.csv,long_longitudinal,2,9,1,2,ok,0 columns with missing values (mean missing fr...,ok
4,longitudinal_wide_template.csv,wide_longitudinal,2,11,1,2,ok,0 columns with missing values (mean missing fr...,ok
5,questionnaires_template.csv,wide_longitudinal,2,9,1,2,ok,0 columns with missing values (mean missing fr...,ok
6,sleep_activity_template.csv,wide_longitudinal,2,10,1,2,ok,0 columns with missing values (mean missing fr...,ok


## 8. Standardized inputs

Standard schema:
```text
subject_id, timepoint, mission_phase, time_index,
variable_name, value, unit, data_stream, source_table
```

In [6]:
std = pd.read_csv(RESULTS_TABLES / 'standardized_longitudinal_variables.csv')
print('standardized rows:', len(std), '| variables:', std['variable_name'].nunique())
std.head(12)

standardized rows: 48 | variables: 24


,subject_id,timepoint,mission_phase,time_index,variable_name,value,unit,data_stream,source_table
0,Demo_Crew_01,T0_baseline,baseline,0,crp,1.0,unknown,biomarker,biomarkers_template
1,Demo_Crew_01,T2_inflight,inflight,2,crp,1.8,unknown,biomarker,biomarkers_template
2,Demo_Crew_01,T0_baseline,baseline,0,glucose,90.0,unknown,biomarker,biomarkers_template
3,Demo_Crew_01,T2_inflight,inflight,2,glucose,102.0,unknown,biomarker,biomarkers_template
4,Demo_Crew_01,T0_baseline,baseline,0,hemoglobin,14.5,unknown,biomarker,biomarkers_template
5,Demo_Crew_01,T2_inflight,inflight,2,hemoglobin,13.8,unknown,biomarker,biomarkers_template
6,Demo_Crew_01,T0_baseline,baseline,0,hematocrit,43.0,unknown,biomarker,biomarkers_template
7,Demo_Crew_01,T2_inflight,inflight,2,hematocrit,41.0,unknown,biomarker,biomarkers_template
8,Demo_Crew_01,T0_baseline,baseline,0,cholesterol,180.0,unknown,biomarker,biomarkers_template
9,Demo_Crew_01,T2_inflight,inflight,2,cholesterol,195.0,unknown,biomarker,biomarkers_template


## 9. Map variables to biological domains

Variable-to-domain mapping is approximate and extensible. Coverage is reported explicitly; unmapped variables are surfaced rather than hidden.

In [7]:
mapping_report = pd.read_csv(RESULTS_TABLES / 'variable_domain_mapping_report.csv')
display(mapping_report)
coverage = pd.read_csv(RESULTS_TABLES / 'domain_coverage_report.csv')
display(coverage)

,variable,normalized_variable,canonical_variable,domain,data_stream,expected_unit,matched_pattern,mapping_status
0,crp,crp,crp,inflammation / immune-adjacent status,biomarker,mg/L,crp,mapped
1,glucose,glucose,glucose,metabolic regulation,biomarker,mg/dL,glucose,mapped
2,hemoglobin,hemoglobin,hemoglobin,hematologic / oxygen-carrying capacity,biomarker,g/dL,hemoglobin,mapped
3,hematocrit,hematocrit,hematocrit,hematologic / oxygen-carrying capacity,biomarker,%,hematocrit,mapped
4,cholesterol,cholesterol,cholesterol,metabolic regulation,biomarker,mg/dL,cholesterol,mapped
5,reaction_time,reaction_time,reaction_time,cognitive load,cognitive,ms,reaction_time,mapped
6,accuracy,accuracy,accuracy,cognitive load,cognitive,%,accuracy,mapped
7,cognitive_score,cognitive_score,cognitive_score,cognitive load,cognitive,score,cognitive_score,mapped
8,co2,co2,co2,environmental context,environmental,ppm,co2,mapped
9,temperature,temperature,temperature,environmental context,environmental,C,temperature,mapped


,domain,mapped_variable_count,variables,data_streams,coverage_status
0,cardiovascular regulation,2,heart_rate; resting_hr,vitals,covered
1,metabolic regulation,2,cholesterol; glucose,biomarker,covered
2,body composition / physical status,0,none,none,absent
3,inflammation / immune-adjacent status,1,crp,biomarker,covered
4,hematologic / oxygen-carrying capacity,2,hematocrit; hemoglobin,biomarker,covered
5,recovery-related markers,0,none,none,absent
6,sleep / circadian regulation,3,sleep_duration; sleep_efficiency; wake_after_s...,sleep_activity,covered
7,autonomic regulation,1,hrv_rmssd,autonomic,covered
8,cognitive load,3,accuracy; cognitive_score; reaction_time,cognitive,covered
9,emotional regulation,2,mood_score; stress_score,questionnaire,covered


## 10. Compute baseline-relative variable deltas

This is the bridge from raw variable values to within-subject trajectory modeling. When no explicit baseline phase exists, the earliest time index is used as baseline and the assumption is recorded.

In [8]:
deltas = pd.read_csv(RESULTS_TABLES / 'variable_baseline_deltas.csv')
deltas.head(12)

,subject_id,timepoint,mission_phase,time_index,variable_name,value,unit,data_stream,baseline_value,delta_from_baseline,percent_change_from_baseline,baseline_assumption
0,Demo_Crew_01,T0_baseline,baseline,0,crp,1.0,unknown,biomarker,1.0,0.0,0.00000,baseline_phase='baseline'
1,Demo_Crew_01,T2_inflight,inflight,2,crp,1.8,unknown,biomarker,1.0,0.8,80.00000,baseline_phase='baseline'
2,Demo_Crew_01,T0_baseline,baseline,0,glucose,90.0,unknown,biomarker,90.0,0.0,0.00000,baseline_phase='baseline'
3,Demo_Crew_01,T2_inflight,inflight,2,glucose,102.0,unknown,biomarker,90.0,12.0,13.33333,baseline_phase='baseline'
4,Demo_Crew_01,T0_baseline,baseline,0,hemoglobin,14.5,unknown,biomarker,14.5,0.0,0.00000,baseline_phase='baseline'
5,Demo_Crew_01,T2_inflight,inflight,2,hemoglobin,13.8,unknown,biomarker,14.5,-0.7,-4.82759,baseline_phase='baseline'
6,Demo_Crew_01,T0_baseline,baseline,0,hematocrit,43.0,unknown,biomarker,43.0,0.0,0.00000,baseline_phase='baseline'
7,Demo_Crew_01,T2_inflight,inflight,2,hematocrit,41.0,unknown,biomarker,43.0,-2.0,-4.65116,baseline_phase='baseline'
8,Demo_Crew_01,T0_baseline,baseline,0,cholesterol,180.0,unknown,biomarker,180.0,0.0,0.00000,baseline_phase='baseline'
9,Demo_Crew_01,T2_inflight,inflight,2,cholesterol,195.0,unknown,biomarker,180.0,15.0,8.33333,baseline_phase='baseline'


## 11. Build graph-ready domain scores

Per-domain aggregation of baseline-relative deltas (default `mean_abs_delta`). The wide output can be used directly by Phase 6 within-subject longitudinal graph trajectory code.

In [9]:
scores_long = pd.read_csv(RESULTS_TABLES / 'adapter_domain_scores_long.csv')
display(scores_long.head(12))
generated = pd.read_csv(RESULTS_TABLES / 'adapter_generated_longitudinal_domain_scores.csv')
print('graph-ready wide columns:', list(generated.columns))
display(generated)

,subject_id,timepoint,mission_phase,time_index,domain,domain_score,domain_score_method,available_variable_count,missing_variable_count,data_streams_used,data_quality_note
0,Demo_Crew_01,T0_baseline,baseline,0,inflammation / immune-adjacent status,0.000000,mean_abs_delta,1,0,biomarker,All 1 domain variables available (self-baselin...
1,Demo_Crew_01,T2_inflight,inflight,2,inflammation / immune-adjacent status,0.800000,mean_abs_delta,1,0,biomarker,All 1 domain variables available (self-baselin...
2,Demo_Crew_01,T0_baseline,baseline,0,metabolic regulation,0.000000,mean_abs_delta,2,0,biomarker,All 2 domain variables available (self-baselin...
3,Demo_Crew_01,T2_inflight,inflight,2,metabolic regulation,13.500000,mean_abs_delta,2,0,biomarker,All 2 domain variables available (self-baselin...
4,Demo_Crew_01,T0_baseline,baseline,0,hematologic / oxygen-carrying capacity,0.000000,mean_abs_delta,2,0,biomarker,All 2 domain variables available (self-baselin...
5,Demo_Crew_01,T2_inflight,inflight,2,hematologic / oxygen-carrying capacity,1.350000,mean_abs_delta,2,0,biomarker,All 2 domain variables available (self-baselin...
6,Demo_Crew_01,T0_baseline,baseline,0,cognitive load,0.000000,mean_abs_delta,3,0,cognitive,All 3 domain variables available (self-baselin...
7,Demo_Crew_01,T2_inflight,inflight,2,cognitive load,14.666667,mean_abs_delta,3,0,cognitive,All 3 domain variables available (self-baselin...
8,Demo_Crew_01,T0_baseline,baseline,0,environmental context,0.000000,mean_abs_delta,5,0,environmental,All 5 domain variables available (self-baselin...
9,Demo_Crew_01,T2_inflight,inflight,2,environmental context,348.300000,mean_abs_delta,5,0,environmental,All 5 domain variables available (self-baselin...


graph-ready wide columns: ['subject_id', 'timepoint', 'mission_phase', 'time_index', 'data_type', 'autonomic regulation', 'cardiovascular regulation', 'cognitive load', 'emotional regulation', 'environmental context', 'hematologic / oxygen-carrying capacity', 'inflammation / immune-adjacent status', 'metabolic regulation', 'recovery capacity', 'sleep / circadian regulation']


,subject_id,timepoint,mission_phase,time_index,data_type,autonomic regulation,cardiovascular regulation,cognitive load,emotional regulation,environmental context,hematologic / oxygen-carrying capacity,inflammation / immune-adjacent status,metabolic regulation,recovery capacity,sleep / circadian regulation
0,Demo_Crew_01,T0_baseline,baseline,0,schema_template_not_scientific_evidence,0.0,0.0,0.000000,0.0,0.0,0.00,0.0,0.0,0.0,0.000000
1,Demo_Crew_01,T2_inflight,inflight,2,schema_template_not_scientific_evidence,13.0,7.0,14.666667,2.5,348.3,1.35,0.8,13.5,3.0,8.766667


### 11a. Confirm Phase 6 compatibility

The generated wide table exposes the required longitudinal index columns plus numeric domain columns, so it is recognized by the Phase 6 code.

In [10]:
from neurobridge_graph.longitudinal import detect_longitudinal_columns
info = detect_longitudinal_columns(generated)
print('required_cols:', info['required_cols'])
print('missing_required:', info['missing_required'])
print('domain_cols:', info['domain_cols'])

required_cols: ['subject_id', 'timepoint', 'mission_phase', 'time_index']
missing_required: []
domain_cols: ['autonomic regulation', 'cardiovascular regulation', 'cognitive load', 'emotional regulation', 'environmental context', 'hematologic / oxygen-carrying capacity', 'inflammation / immune-adjacent status', 'metabolic regulation', 'recovery capacity', 'sleep / circadian regulation']


## 12. Adapter report

Saved to `results/reports/phase10_data_adapter_report.txt`.

In [11]:
report_text = (RESULTS_REPORTS / 'phase10_data_adapter_report.txt').read_text()
print(report_text)

NeuroBridge-S4 — Phase 10 HRP-Like Data Adapter Report

Phase 10 does not interpret health status. It validates and transforms HRP-like longitudinal data streams into a graph-ready self-baseline schema for downstream NeuroBridge-S4 analysis.

Data source
-----------

No real HRP-like input files were found. Schema templates were used to demonstrate adapter mechanics only. Template data are not scientific evidence.

Input tables
------------

- biomarkers_template.csv: format=wide_longitudinal, rows=2, columns=10, subjects=1, timepoints=2, required=ok
- cognitive_tests_template.csv: format=wide_longitudinal, rows=2, columns=8, subjects=1, timepoints=2, required=ok
- environmental_context_template.csv: format=wide_longitudinal, rows=2, columns=10, subjects=1, timepoints=2, required=ok
- longitudinal_long_template.csv: format=long_longitudinal, rows=2, columns=9, subjects=1, timepoints=2, required=ok
- longitudinal_wide_template.csv: format=wide_longitudinal, rows=2, columns=11, subjects=

## 13. Conclusion

Phase 10 makes NeuroBridge-S4 more realistic as a platform by separating data ingestion and validation from graph analysis. HRP-like multimodal streams can be validated, mapped to biological domains, transformed against each subject's own baseline, and exported as graph-ready domain scores for the within-subject longitudinal trajectory pipeline.

This adapter validates and transforms HRP-like longitudinal data into graph-ready domain scores. It does not diagnose, score risk, infer exposure, or recommend treatment.